In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.utils import resample
from joblib import Parallel, delayed
import multiprocessing

# Define the theta function
def theta(df):
    model = DecisionTreeRegressor(max_depth=5)
    model.fit(df[['x1']], df['y'])
    return model.predict([[0.35]])[0]

# Define the bootstrap function
def bootstrap(indices, B, theta, df):
    thetastar = []
    for b in range(B):
        sample_indices = resample(indices, replace=True)
        sample_df = df.iloc[sample_indices]
        thetastar.append(theta(sample_df))
    return np.array(thetastar)

# Simulation parameters
z = []
n = 500
B = 500  # Paper: 10000
nsim = 1000  # Paper: 1000

# Set up parallel processing
num_cores = multiprocessing.cpu_count() - 1

# Main simulation loop
def simulate(sim):
    np.random.seed(sim)  # For reproducibility
    x1 = np.random.uniform(0, 1, n)
    e = np.random.normal(0, np.sqrt(2), n)
    
    y = (0.7 * ((x1 >= 0.35) & (x1 < 0.45)) +
         0.7 * ((x1 >= 0.55) & (x1 < 0.65)) +
         1.4 * ((x1 >= 0.45) & (x1 < 0.55)) +
         e)
    
    df = pd.DataFrame({'x1': x1, 'y': y})
    
    p = bootstrap(np.arange(n), B, theta, df)
    return np.mean(p)

results = Parallel(n_jobs=num_cores)(delayed(simulate)(sim) for sim in range(nsim))

# Calculate the summary statistics
mean_z = np.mean(results)
var_z = np.var(results)

# Output the results
print(f"Mean: {mean_z}")
print(f"Variance: {var_z}")
